In [ ]:
from _client import Client
import httpx
import json

In [ ]:
base_url="https://freva.dkrz.de/api/chatbot/"

In [ ]:
url = base_url + "ping"
response=httpx.get(url=url)
print(response.json())

In [ ]:
auth_key="***REMOVED***"
input_string = "Please write some Python code to plot the global yearly average temperatures between the years 1980 and 2010. Thank you."

In [ ]:
url = base_url + "streamresponse"

raw_response = []
with httpx.stream(method="GET", url=url, timeout=None, params={"auth_key":auth_key, "input":input_string, "thread_id":  None}) as r:
    for chunk in r.iter_bytes():
        raw_response.append(chunk)

In [ ]:
def process_chunks(chunk:str, partial_response:str=""):
    chunk_split = chunk.split("}{")
    if len(chunk_split) == 1:
        if chunk[0] == "{" and chunk[-1] == "}":
            return [chunk], ""
        elif chunk[0] == "{" and chunk[-1] != "}":
            partial_response = chunk
            return [], partial_response
        elif chunk[-1] == "}":
            partial_response += chunk
            return [partial_response], ""
        else:
            partial_response += chunk
            return [], partial_response
    else:
        complete_parts = []
        for i, part in enumerate(chunk_split):
            if i==0:
                fixed_part = part + "}"
                if part[0] != "{":
                    partial_response += fixed_part
                    complete_parts.append(partial_response)
                    continue
            elif i==len(chunk_split)-1:
                fixed_part = "{" + part
                if part[-1] != "}": 
                    partial_response = fixed_part
                    return complete_parts, partial_response
                complete_parts.append(fixed_part)
                return complete_parts, ""
            else:
                fixed_part = "{" + part + "}"
            complete_parts.append(fixed_part)    

In [ ]:
complete_response = []
partial_response=""
for i, chunk in enumerate(raw_response):
    chunk_decoded=chunk.decode("utf-8")
    complete_parts, partial_response = process_chunks(chunk_decoded, partial_response)
    complete_response += complete_parts

In [ ]:
processed_response=[]
for i, r in enumerate(complete_response):
    result=json.loads(r)
    processed_response.append(result)

In [ ]:
thread_id = json.loads(processed_response[0]["content"])["thread_id"]

In [ ]:
url = base_url + "streamresponse"
input_string="Please execute this code now."
raw_response = []
with httpx.stream(method="GET", url=url, timeout=None, params={"auth_key":auth_key, "input":input_string, "thread_id": thread_id}) as r:
    for chunk in r.iter_bytes():
        raw_response.append(chunk)

In [ ]:
complete_response = []
partial_response=""
for i, chunk in enumerate(raw_response):
    chunk_decoded=chunk.decode("utf-8")
    complete_parts, partial_response = process_chunks(chunk_decoded, partial_response)
    complete_response += complete_parts

In [ ]:
for i, r in enumerate(complete_response):
    result=json.loads(r)
    processed_response.append(result)

In [ ]:
for r in processed_response:
    print(r)

In [ ]:
with open("example_conversation.json", "w") as fo:
    json.dump(processed_response, fo, indent=2)